# Python Alpha Research

1. Edit **CONFIG** below: where fields come from, which research directions, how many alphas.
2. Run all cells. The system picks fields, cleans and backfills them, writes the Python code, checks it, simulates and ranks the results.
3. Read the results, then run **Refine** to improve the best alphas.

You never write Python. Ideas are written like Fast Expressions (or not at all: pick a direction, or describe an idea for the AI).

In [ ]:
CONFIG = dict(
    # ---------------------------------------------------------------- where fields come from
    region="USA", delay=1, universe="TOP3000",
    sources=[
        {"dataset": "pv1", "select": "all"},
        {"dataset": "fundamental6", "prefix": "fnd6_", "select": "least_used", "n": 30},
        # {"dataset": "analyst4", "contains": ["eps", "rev"], "select": "random", "n": 20, "seed": 1},
        # {"name": "prices", "fields": ["close", "vwap", "volume", "cap"]},
    ],
    # select: "all" | "random" | "least_used" | "most_used" | "top_coverage";  n = how many
    # filters: prefix, contains, regex, exclude, min_coverage, search

    # ---------------------------------------------------------------- what to try (combine freely)
    directions=["reversion", "value"],   # see rs.directions(); [] for none
    expressions=[                        # your own ideas, FE syntax; X/Y/Z are filled from the pool
        # "ts_rank(rank(X), 20)",
        # {"expr": "rank(X / Y)", "roles": {"X": "fundamental", "Y": "fundamental"}},
        # {"expr": "ts_rank(rank(returns), 20)", "fe_alpha_id": "abc123"},   # translate + tag an FE alpha
    ],
    roles={},                            # default roles for X/Y in expressions, e.g. {"X": "analyst@analyst4"}
    compose=None,                        # operator grid, e.g.
    # compose={"start": ["X"], "layers": [["rank(_)", "zscore(_)"], ["ts_rank(_, 20)", "ts_decay_linear(_, 10)"], ["_", "-_"]]},
    ai_idea="",                          # plain words, e.g. "firms with improving cash flow relative to peers"

    # ---------------------------------------------------------------- how many, which settings
    max_alphas=50,                       # always generates exactly this many new ones
    settings={                           # a list = try each value
        "neutralization": ["MARKET", "SECTOR"],
        "universe": ["TOP3000", "TOP1000"],
        # "decay": [0, 4],
    },
    settings_mode="spread",              # spread: rotate combos | all: every alpha x every combo | random
    concurrency=3,                       # simulations running at the same time (1 to 8)
    batch_size=10,                       # results are saved and summarized after every batch

    # ---------------------------------------------------------------- cleaning and operators
    prep="auto",                         # minimal | auto | robust | {"fundamental": ["auto", "winsorize(std=3)"]}
    allowed_ops="all",                   # or e.g. ["time-series", "cross-sectional", "group"] / op names
    blocked_ops=[],                      # e.g. ["ml", "stateful", "ts_pca_neutralize"]

    # ---------------------------------------------------------------- AI (optional)
    llm={},                              # leave empty to use ~/secrets/llm.json or env ANTHROPIC_API_KEY / OPENAI_API_KEY
    offline=False,                       # True = no BRAIN calls (build + check only)
)

To store an AI key once (skip if you don't use AI): `import llm; llm.save_llm_config('anthropic', 'sk-ant-...')` (or `'openai'`, or `'openai_compatible'` with `base_url=`).

## 1. Connect and load fields

In [ ]:
import os, sys; sys.path.insert(0, os.getcwd())
from research import Research
rs = Research(CONFIG).connect()
rs.load_fields()
rs.fields()

## 2. (Optional) Browse directions and operators

In [ ]:
rs.directions()

In [ ]:
rs.ops()            # all operators; try rs.ops("volatility") or rs.ops(category="group")

## 3. (Optional) AI ideas from your `ai_idea` text

In [ ]:
if CONFIG["ai_idea"]:
    rs.ai_ideas()   # without a key it prints a prompt; paste the reply into rs.paste_ai_reply("""...""")

## 4. Generate

In [ ]:
rs.generate()

In [ ]:
rs.show_code(0)     # the exact Python submitted for alpha #0

## 5. Simulate

In [ ]:
rs.simulate()

## 6. Results (all rounds, best first)

In [ ]:
rs.results().head(20)

## 7. Refine the best and simulate again

In [ ]:
rs.refine(top=5)
rs.simulate()

In [ ]:
# rs.ai_refine()    # optional: the AI proposes new ideas from your results; then rs.generate(); rs.simulate()

## 8. Save the best (code files for the platform UI)

In [ ]:
rs.save_best(5)

---
### Shortcuts
* Everything in one go: `rs.run(rounds=2)`
* New operator from a description: `rs.ai_operator("ratio of up-volume to down-volume over d days")`, then use it in expressions
* Change a field's kind: `CONFIG["kinds"] = {"fnd6_xyz": "fundamental"}`, re-run from step 1

### Roles for X / Y / Z
`"any"`, a kind (`price, volume, returns, fundamental, analyst, event, model, sparse, other`), `"dense"`, `"slow"`, `"@source"` (any field from that source), `"kind@source"`, or a list of fields.

### Prep (automatic cleaning)
Every field: missing-value sentinels and inf become NaN. `auto` adds split adjustment for per-share prices, forward-fill for fundamentals (250 days), analyst/sparse (66), models (22), news (5). `robust` also winsorizes. A dict sets steps per field, kind, `@source` or `*`.